# 02 — Diagnostic Workflow

This notebook demonstrates the **diagnostic** use-case:  
a single-run ISA-JSON where each assay holds exactly one measurement file.

All plots are interactive **Bokeh** figures: hover for exact values, zoom, pan, and save with the toolbar.

**Covers:**
1. Load ISAWrapper (single-run fixture)
   - 1b. Sensor inventory — all channels at a glance
   - 1c. Test matrix — factors × sensors design table
2. Navigate study → assay
3. Load signal DataFrame
   - 3b. Plot the signal (time domain)
   - 3c. Outlier detection & fixing (`strategy`: clip · nan · drop · **interpolate · ffill · bfill**)
   - 3d. Missing-values report
   - 3e. Filling outliers by interpolation
4. Distribution plots — all sensors (grouped by measurement type)
5. Frequency-domain plots — all sensors (grouped by measurement type)
6. Cross-sensor comparison — interactive boxplot
7. Variable overview (factor × assay matrix)
8. Fault labels — bridge to machine learning
9. Protocol parameters — measurement & processing settings

In [1]:
%pip install -q pydantic pandas numpy scipy bokeh --quiet

Note: you may need to restart the kernel to use updated packages.


In [14]:
import sys
import logging
from pathlib import Path

# Add python-wrapper package to path
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Suppress noisy @id resolution warnings from ISA-tools
logging.getLogger("isa_phm").setLevel(logging.ERROR)

import pandas as pd

from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()         # enable Bokeh inline rendering in the notebook

from isa_phm import ISAWrapper
from isa_phm.errors import DataFileError

ISA_JSON = Path("G:\\ISA\\Datasets\\Pepijn-Knarskast-Diagnostic\\Knarskast Test Measurement ISA-PHM.json").resolve()
print(f"Fixture : {ISA_JSON}")
print(f"Exists  : {ISA_JSON.exists()}")

Loading BokehJS ...

Fixture : G:\ISA\Datasets\Pepijn-Knarskast-Diagnostic\Knarskast Test Measurement ISA-PHM.json
Exists  : True


## 1. Load the wrapper

In [15]:
wrapper = ISAWrapper(
    path=ISA_JSON,
    strict_validation=False,
    auto_fix=True,
)

ov = wrapper.investigation_overview()
print(f"Investigation : {ov.title}")
print(f"Studies       : {ov.n_studies}")
print(f"Experiment    : {ov.experiment_type}")

Investigation :  Embedded fiberoptic sensing for progressive tooth breakage detection in planetary gear systems
Studies       : 1
Experiment    : diagnostic-experiment


## 1b. Sensor inventory

`study.list_assays()` returns one `AssaySummary` per sensor channel.  
Displaying it as a DataFrame gives a quick inventory of all sensors in this study.


In [16]:
studies = wrapper.list_studies()
first_study_title = studies[0].title
study = wrapper.study(first_study_title)

# Build sensor inventory table
assay_summaries = study.list_assays()
sensor_df = pd.DataFrame([
    {
        "assay_id":          s.assay_id,
        "sensor_alias":      s.sensor_alias,
        "technology_type":   s.technology_type,
        "measurement_type":  s.measurement_type,
        "n_runs":            s.n_runs,
        "n_raw_files":       s.n_raw_files,
        "n_processed_files": s.n_processed_files,
    }
    for s in assay_summaries
])

print(f"Study  : {first_study_title}")
print(f"Sensors: {len(sensor_df)}")
display(sensor_df)


Study  : Diagnostic sensor test
Sensors: 8


,assay_id,sensor_alias,technology_type,measurement_type,n_runs,n_raw_files,n_processed_files
0,a_st01_se01,FBG Sensor 1,FBG Sleeve,Wavelength,1,1,0
1,a_st01_se02,FBG Sensor 2,FBG Sleeve,Wavelength,1,1,0
2,a_st01_se03,FBG Sensor 3,FBG Sleeve,Wavelength,1,1,0
3,a_st01_se04,FBG Sensor 4,FBG Sleeve,Wavelength,1,1,0
4,a_st01_se05,FBG Sensor 5,FBG Sleeve,Wavelength,1,1,0
5,a_st01_se06,FBG Sensor 6,FBG Sleeve,Wavelength,1,1,0
6,a_st01_se07,FBG Sensor 7,FBG Sleeve,Wavelength,1,1,0
7,a_st01_se08,FBG Sensor 8,FBG Sleeve,Wavelength,1,1,0


## 1c. Test matrix

`study.test_matrix()` returns a **pivot table** matching the ISA-PHM-Wizard test matrix view:  
rows are study variables (fault specifications first, then operating conditions),  
columns are the unique experimental conditions tested.

`study.fault_conditions()` and `study.operating_conditions()` return the same pivot  
filtered to the respective variable category.

`study.list_factors()` lists the raw factor definitions for this study.


In [17]:
# Filtered views
print("\nFault conditions:")
display(study.fault_conditions())

print("\nOperating conditions:")
display(study.operating_conditions())

# Full test matrix (fault specs first, then operating conditions)
print("Test matrix:")
display(study.test_matrix())



Fault conditions:


,variable,type,unit,Value
0,Fault Severity,Quantitative fault specification,,0



Operating conditions:


,variable,type,unit,Value
0,Motor speed (Hz),Operating condition,Hz,10 Hz


Test matrix:


,variable,type,unit,Value
0,Fault Severity,Quantitative fault specification,,0
1,Motor speed (Hz),Operating condition,Hz,10 Hz


## 2. Navigate study → assay

In [18]:
# study, assay_summaries already defined in section 1b
print(f"Study : {study.title}")

first_assay_id = assay_summaries[0].assay_id
assay = study.assay(first_assay_id)

assay_ov = assay.overview()
print(f"Assay : {assay_ov.assay_id}")
print(f"Sensor: {assay_ov.sensor_alias}  |  Measurement: {assay_ov.measurement_type}")
print(f"Runs  : {assay_ov.n_runs}")


Study : Diagnostic sensor test
Assay : a_st01_se01
Sensor: FBG Sensor 1  |  Measurement: Wavelength
Runs  : 1


## 3. Load signal data

`assay.load_dataframe()` reads the CSV and attaches study/assay/run metadata columns.

**`file_type` parameter** — choose which data file to load:

| Value | When to use |
|-------|-------------|
| `"processed"` *(default)* | ISA-JSON has a processing step that outputs a derived file |
| `"raw"` | ISA-JSON only records raw measurement files (typical for diagnostic datasets) |

If the requested type is not available the other type is used automatically and a warning is logged.
The fixture paths point to an external drive — the cell handles the missing file gracefully.


In [ ]:
df = None

# Pass file_type="raw" when the ISA-JSON only records raw measurement files.
# Omit file_type (or use "processed") when a processing step is present.
df = assay.load_dataframe(file_type="raw")
print(f"Loaded {len(df)} rows × {len(df.columns)} cols")
print("Columns:", list(df.columns))
display(df.head(10))


Loaded 90388 rows × 2 cols
Columns: ['time', 'value']


,time,value
0,0.000000,1515.822906
1,0.000053,1515.845703
2,0.000105,1515.796814
3,0.000158,1516.005005
4,0.000211,1515.822083
5,0.000263,1515.840485
6,0.000316,1515.793518
7,0.000368,1515.739960
8,0.000421,1515.829224
9,0.000474,1515.877838


: 

## 3b. Plot the signal (time domain)

`assay.plot_timeseries()` draws the raw waveform as an interactive Bokeh figure.  
The y-axis label is automatically set from the ISA-JSON measurement type and unit (e.g. *Wavelength measurement (nm)*).

Use the toolbar (top-right of the figure) to **pan**, **zoom**, or **save** the plot.  
Hover over the line to read exact time and amplitude values.

Pass `show_outliers=True` to highlight flagged samples in red in the same call.

In [8]:
fig = assay.plot_timeseries(file_type="raw")
bokeh_show(fig)

## 3c. Outlier detection & fixing

| Method | Description |
|--------|-------------|
| `assay.detect_outliers(method, upper)` | Returns an `OutlierReport`; call `.to_dataframe()` to display |
| `assay.plot_timeseries(show_outliers=True)` | Waveform with outliers highlighted in red |
| `assay.fix_outliers(df, method, upper, strategy)` | Returns a corrected copy of the DataFrame |
| `assay.plot_outlier_comparison(df, df_clean)` | Before/after waveform side by side |

**`method`** — `"iqr"` (robust, statistical) · `"zscore"` (mean-based) · `"fixed"` (explicit bounds)  
**`upper` / `lower`** — hard value thresholds, e.g. `upper=1e7` to remove only sensor overflow values  
**`strategy`** — `"clip"` · `"nan"` · `"drop"` · `"interpolate"` · `"ffill"` · `"bfill"`

> **Tip:** use `method="fixed", upper=<limit>` when the signal has known hardware sentinel values  
> (e.g. `4294967295` = 2³² − 1, a common uint32 overflow marker from FBG interrogators).

> **`strategy` quick guide**  
> * `"drop"` — remove outlier rows; best when a clean record length is needed.  
> * `"clip"` — clamp to detection bounds; shape is preserved, extremes softened.  
> * `"interpolate"` — linear interpolation across flagged samples; smooth reconstruction.  
> * `"ffill"` / `"bfill"` — carry the last (or next) good sample forward (or backward).

In [9]:
# Use method="fixed" with upper=1e7 to flag only sensor overflow values
# (4294967295 = 2^32−1 is a uint32 sentinel from some FBG interrogators)
report = assay.detect_outliers(file_type="raw", method="fixed", upper=1e7)
display(report.to_dataframe())

# Waveform with overflow samples highlighted in red
fig = assay.plot_timeseries(
    file_type="raw",
    show_outliers=True,
    outlier_method="fixed",
    outlier_upper=1e7,
)
bokeh_show(fig)

,column,n_outliers,pct_outliers,lower_bound,upper_bound,method,threshold
0,value,778,0.861,-inf,10000000.0,fixed,3.0


In [10]:
# Drop rows where the value is a hardware overflow sentinel (> 1e7)
# strategy="drop" removes those rows entirely; valid signal is untouched
df_clean = assay.fix_outliers(df, method="fixed", upper=1e7, strategy="drop")

print(f"Rows before : {len(df)}      | Rows after : {len(df_clean)}")
print(f"Range before: [{df['value'].min():.4f}, {df['value'].max():.4f}]")
print(f"Range after : [{df_clean['value'].min():.4f}, {df_clean['value'].max():.4f}]")

fig = assay.plot_outlier_comparison(df, df_clean, strategy="drop")
bokeh_show(fig)

Rows before : 90388      | Rows after : 89610
Range before: [1515.4266, 4294967295.0000]
Range after : [1515.4266, 1516.1602]


## 3d. Missing-values report

`assay.missing_values_report()` scans the raw DataFrame for `NaN` values and returns a `MissingValuesReport`.  
Run it **before** training or analysis to catch any gaps introduced by sensors going offline or data-acquisition dropout.

In [11]:
from isa_phm.schemas import MissingValuesReport

report = assay.missing_values_report(file_type="raw")

print(f"Shape          : {report.n_rows} rows × {report.n_cols} cols")
print(f"Total missing  : {report.n_missing}  ({report.pct_missing:.1f}%)")
print()
by_col = pd.Series(report.by_column, name="missing")
display(by_col[by_col > 0] if by_col.any() else "No missing values.")


Shape          : 90388 rows × 2 cols
Total missing  : 0  (0.0%)



'No missing values.'

## 3e. Filling outliers by interpolation

When continuity of the time-series matters — e.g. for FFT or envelope analysis — linear interpolation gives a smoother result than clipping or dropping.

Available gap-fill strategies:

| `strategy` | Behaviour |
|---|---|
| `"interpolate"` | Linear interpolation between flanking valid samples |
| `"ffill"` | Forward-fill — propagates the last valid sample forward |
| `"bfill"` | Backward-fill — propagates the next valid sample backward |

In [11]:
# Linear interpolation: replace overflow sentinels with smooth estimated values
df_interp = assay.fix_outliers(df, method="fixed", upper=1e7, strategy="interpolate")

print(f"NaN remaining after interpolate : {df_interp['value'].isna().sum()}")
print(f"Range after fix : [{df_interp['value'].min():.4f}, {df_interp['value'].max():.4f}]")

# Forward-fill variant
df_ffill = assay.fix_outliers(df, method="fixed", upper=1e7, strategy="ffill")
print(f"NaN remaining after ffill       : {df_ffill['value'].isna().sum()}")

# Side-by-side comparison (uses the interpolated version as the "clean" signal)
fig = assay.plot_outlier_comparison(df, df_interp, strategy="interpolate")
bokeh_show(fig)

NaN remaining after interpolate : 0
Range after fix : [1515.4266, 1516.1602]
NaN remaining after ffill       : 0


## 4. Distribution & FFT — all sensors

Sections 4 and 5 loop over every sensor channel.  
Each iteration loads, cleans, plots, then discards — only one sensor is in memory at a time.  
Sensors are automatically **grouped by measurement type** so that channels measuring the same physical quantity appear together.

In [ ]:
# Distribution — all sensors, grouped by measurement type, one sensor at a time
from bokeh.models import Div

assay_summaries = study.list_assays()

groups: dict[str, list] = {}
for s in assay_summaries:
    groups.setdefault(s.measurement_type, []).append(s)

for mtype, group_summaries in groups.items():
    bokeh_show(Div(text=f"<h3>{mtype}</h3>"))
    for summary in group_summaries:
        a = study.assay(summary.assay_id)
        df = a.load_dataframe(file_type="raw")
        df_clean = a.fix_outliers(df, method="fixed", upper=1e7, strategy="drop")
        fig = a.plot_distribution(df_clean)
        bokeh_show(fig)
        del df, df_clean          # free memory before loading the next sensor

## 5. Frequency-domain — all sensors

Same grouping by measurement type as section 4.

In [12]:
# FFT — all sensors, grouped by measurement type, one sensor at a time
for mtype, group_summaries in groups.items():
    bokeh_show(Div(text=f"<h3>{mtype}</h3>"))
    for summary in group_summaries:
        a = study.assay(summary.assay_id)
        df = a.load_dataframe(file_type="raw")
        df_clean = a.fix_outliers(df, method="fixed", upper=1e7, strategy="drop")
        fig = a.plot_frequency_domain(df_clean, log_scale=False)
        bokeh_show(fig)
        del df, df_clean          # free memory before loading the next sensor

NameError: name 'groups' is not defined

## 6. Cross-sensor comparison — interactive boxplot

A **Bokeh** box plot with one box per sensor channel — all channels on a single figure.

**What you can do:**
- **Hover** over a box → tooltip shows median, mean, Q1, Q3, and whisker bounds
- **Scroll** to zoom in/out on the y-axis
- **Drag** to pan; use *Box Zoom* in the toolbar for a precise region
- **Click** the disc icon in the toolbar to save as PNG

**How to read the boxes:**
- Dark blue = Q1 → median (lower half of the IQR)
- Light blue = median → Q3 (upper half of the IQR)
- Whiskers extend to 1.5 × IQR (or data min/max, whichever comes first)
- **Orange dot** = mean

Only summary statistics are held in memory after loading — not the full arrays.


In [ ]:
# Cross-sensor boxplot — all sensors on a single interactive figure.
fig = study.plot_sensor_boxplot(
    file_type='raw',
    outlier_method='fixed',
    outlier_upper=1e7,
    outlier_strategy='drop',
)
bokeh_show(fig)

## 7. Variable overview (factor × assay matrix)

`study.variable_overview()` shows how factors vary across every assay in the study —  
useful for checking design completeness before starting analysis.

In [ ]:
conditions = study.variable_overview()

for i, df in enumerate(conditions, 1):
    if len(conditions) > 1:
        print(f"Condition {i}")
    display(df)

## 8. Fault labels — bridge to machine learning

`study.get_fault_labels()` extracts every **fault-related factor** (factor type
matching `fault`, `damage`, or `rul`) into a tidy DataFrame keyed by `assay_id`
and `run_id`.

Merge it directly with `assay.lifecycle_features()` to get a labelled feature matrix:

```python
features = assay.lifecycle_features()   # signal statistics per run
labels   = study.get_fault_labels()     # fault severity per run
df       = features.merge(labels, on=["assay_id", "run_id"])
```

- **Diagnostic dataset**: fixed fault level — every run gets the same label.
- **Prognostic dataset**: label column holds the evolving damage level / RUL.

> **No fault factors defined?** `get_fault_labels()` returns an empty DataFrame
> with the correct columns — safe to call on any dataset.


In [13]:
# Fault-related factor values for every (assay, run) pair in this study
labels = study.get_fault_labels()
display(labels)

# Optionally filter to a single sensor channel:
# labels_se01 = study.get_fault_labels(assay_id='a_st01_se01')

# Merge with lifecycle features for a ready-to-train DataFrame:
# features = assay.lifecycle_features()
# training_df = features.merge(labels, on=['assay_id', 'run_id'])
# display(training_df.head())

,assay_id,run_id,run_number,Fault Severity
0,a_st01_se01,run_01,1,0
1,a_st01_se02,run_01,1,0
2,a_st01_se03,run_01,1,0
3,a_st01_se04,run_01,1,0
4,a_st01_se05,run_01,1,0
5,a_st01_se06,run_01,1,0
6,a_st01_se07,run_01,1,0
7,a_st01_se08,run_01,1,0


## 9. Protocol parameters — measurement & processing settings

`assay.list_measurement_params()` returns a one-row-per-parameter DataFrame of  
the acquisition settings stored in the ISA-JSON Measurement Protocol  
(e.g. sampling rate, hardware range, sensor model).

`assay.list_processing_params()` returns the same structure for the  
Processing Protocol (e.g. filter type, window size, normalisation method).

Both methods read from the **first run** in the assay and return an empty  
DataFrame (with correct columns) when no parameters are defined.

In [38]:
# Acquisition settings declared in the Measurement Protocol
meas_params = assay.list_measurement_params()
print("Measurement parameters:")
display(meas_params)

# Post-processing settings declared in the Processing Protocol
proc_params = assay.list_processing_params()
print("Processing parameters:")
display(proc_params)

AttributeError: 'AssayProxy' object has no attribute 'list_measurement_params'

## 10. Sensor inventory — catalog & per-sensor metadata

`study.sensor_catalog()` gives a one-row-per-channel overview of every sensor
registered in the study, including inferred sampling rate and measurement unit.

`assay.sensor_info()` returns the same data **plus** the raw protocol parameter
list and the names of every factor associated with that assay — useful for
scripting and data validation.


In [ ]:
# ── Sensor catalog ─────────────────────────────────────────────────────────
catalog = study.sensor_catalog()
print(f"Sensor catalog — {len(catalog)} channel(s) in this study:")
display(catalog)

# ── Per-sensor detail ───────────────────────────────────────────────────────
info = assay.sensor_info()
print("\nDetailed sensor info for the first assay:")
for key, val in info.items():
    print(f"  {key:25s}: {val}")


## 11. Fill missing values

`assay.fill_missing_values(df, strategy=...)` fills NaN gaps in any DataFrame
column using one of five strategies:

| `strategy`      | Behaviour                                           |
|-----------------|-----------------------------------------------------|
| `"interpolate"` | Linear interpolation between valid neighbours       |
| `"ffill"`       | Forward-fill from the last valid sample             |
| `"bfill"`       | Backward-fill from the next valid sample            |
| `"mean"`        | Replace with column mean                            |
| `"zero"`        | Replace with 0                                      |

This is distinct from `fix_outliers()` which operates on _detected anomalies_,
not on structural gaps in the data.


In [ ]:
# Load raw data (may contain NaN from the csv or from outlier-to-NaN conversion)
df_raw = assay.load_dataframe(file_type="raw")

# Artificially introduce NaN in a small slice for demonstration
import numpy as np
df_demo = df_raw.copy()
df_demo.loc[50:60, "value"] = np.nan
print(f"NaN count before fill : {df_demo['value'].isna().sum()}")

# Apply linear interpolation
df_filled = assay.fill_missing_values(df_demo, strategy="interpolate")
print(f"NaN count after fill  : {df_filled['value'].isna().sum()}")
print(f"Value range            : [{df_filled['value'].min():.4f},  {df_filled['value'].max():.4f}]")


## 12. Cross-sensor DataFrame (time-aligned)

`study.load_multi_sensor_dataframe()` loads every sensor channel for the same
run and performs a **nearest-neighbour time merge**, returning a wide DataFrame
with one column per sensor alias alongside the shared `time` column.

This enables:
- Cross-sensor correlation analysis
- Feature construction that combines multiple physical signals
- Input preparation for multivariate ML models


In [ ]:
# Load all sensors for this study and time-align them into one wide DataFrame
# (pass sensors=[alias1, alias2] to select a subset)
multi_df = study.load_multi_sensor_dataframe(file_type="raw")

print(f"Wide DataFrame shape    : {multi_df.shape}")
print(f"Columns                 : {list(multi_df.columns)}")
display(multi_df.head(5))


## 13. ML-ready labeled export

`study.export_labeled_dataset()` merges **every** sensor channel and run into
a single tidy DataFrame, annotating each sample with its sensor identity and
experimental condition.  This is the primary bridge between the ISA-PHM wrapper
and downstream ML pipelines.

Returned columns:

| Column | Description |
|--------|-------------|
| `time` | Sample timestamp (seconds) |
| `value` | Measurement value |
| `assay_id` | Source assay identifier |
| `sensor_alias` | Human-readable sensor name |
| `measurement_type` | e.g. `"Vibration"` |
| `run_id` | Run identifier |
| `run_number` | 1-based run index |
| `<factor>…` | One column per study factor (fault type, severity, speed, …) |

Optional parameters allow inline outlier removal before export.


In [ ]:
# Export every sensor channel × every run, labeled with ISA-PHM metadata
labeled_df = study.export_labeled_dataset(file_type="raw")

print(f"Labeled dataset shape   : {labeled_df.shape}")
print(f"Columns                 : {list(labeled_df.columns)}")
print(f"\nUnique sensors          : {labeled_df['sensor_alias'].unique().tolist()}")
print(f"Unique runs/sensor      : {labeled_df.groupby('sensor_alias')['run_id'].nunique().to_dict()}")
display(labeled_df.head(5))


---

## What's next?

| Notebook | Topic |
|---|---|
| `01_getting_started.ipynb` | API tour — investigation, study, assay navigation |
| `03_prognostic_workflow.ipynb` | Multi-run lifecycle features, trend plots, correlation |